In [1]:
import re
import pandas as pd

In [4]:
merged = pd.read_csv(r"C:\Users\hp\Documents\Montra_Amdari\data\processed\merged_raw.csv")
merged.head()

,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,room_type,booking_channel,...,total_amount,stay_duration,customer_type,loyalty_status,country,previous_stays,hotel_name,location,star_rating,total_rooms
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,Suite,Direct Website,...,296.44,1,Leisure,NaN,France,4,The Aberdeen Grand,Aberdeen,4,118
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,Deluxe,Corporate Portal,...,483.56,4,Business,Gold,Germany,4,Nottingham Park Hotel,Nottingham,4,95
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,Executive,Travel Agent,...,415.59,2,Business,NaN,Ireland,4,The London Boutique,London,4,231
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative,Family Room,Airbnb,...,601.18,3,Leisure,NaN,Nigeria,0,Manchester Metropolitan,Manchester,3,101
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,Deluxe,Expedia,...,603.85,5,Group,NaN,United States,9,Glasgow Metropolitan,Glasgow,4,214


In [5]:
merged.columns

Index(['review_id', 'booking_id', 'guest_id', 'hotel_id', 'review_text',
       'rating_score', 'review_date', 'sentiment_label', 'room_type',
       'booking_channel', 'check_in_date', 'check_out_date', 'total_amount',
       'stay_duration', 'customer_type', 'loyalty_status', 'country',
       'previous_stays', 'hotel_name', 'location', 'star_rating',
       'total_rooms'],
      dtype='str')

### Data Preprocessing

In [6]:
##Inspect the quality and characteristics of the review corpus
print("missing_review_text:", merged['review_text'].isna().sum())
print("Duplicate reviews:", merged['review_text'].duplicated().sum())

missing_review_text: 0
Duplicate reviews: 464


In [7]:
### Remove duplicate reviews
review_corpus = merged.drop_duplicates(
    subset= ['review_text']
).reset_index(drop = True)

In [8]:
## Remove all record that contains whitespaces
review_corpus = review_corpus[
    review_corpus['review_text'].str.strip().ne("")
].copy()

In [9]:
review_corpus['text_length'] = review_corpus['review_text'].str.len()

In [10]:
review_corpus['text_length']

0         80
1        160
2        102
3        129
4        196
        ... 
34531    193
34532    144
34533    200
34534    136
34535    150
Name: text_length, Length: 34536, dtype: int64

In [11]:
review_corpus.head()

,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,room_type,booking_channel,...,stay_duration,customer_type,loyalty_status,country,previous_stays,hotel_name,location,star_rating,total_rooms,text_length
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,Suite,Direct Website,...,1,Leisure,NaN,France,4,The Aberdeen Grand,Aberdeen,4,118,80
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,Deluxe,Corporate Portal,...,4,Business,Gold,Germany,4,Nottingham Park Hotel,Nottingham,4,95,160
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,Executive,Travel Agent,...,2,Business,NaN,Ireland,4,The London Boutique,London,4,231,102
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative,Family Room,Airbnb,...,3,Leisure,NaN,Nigeria,0,Manchester Metropolitan,Manchester,3,101,129
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,Deluxe,Expedia,...,5,Group,NaN,United States,9,Glasgow Metropolitan,Glasgow,4,214,196


In [12]:
def clean_review(text):
    # convert  to string
    text = str(text)
    # remove all tags
    text = re.sub(r'<.*?>',' ', text)
    # remove all url
    text = re.sub(r'https?://\S+|www\.\S+',' ', text)
    ## remove all emojis
    text = text.encode('ascii', 'ignore').decode('ascii')
    #convert to lowercase
    text = text.lower()
    # keep letters, number and spaces
    text = re.sub(r'[^a-z0-9\s]',' ',text)
    # remove extra whitespace
    text = re.sub(r'\s+',' ', text).strip()

    return text

### Apply Preprocessing

In [13]:
merged['cleaned_review_text'] = merged['review_text'].map(clean_review)

In [15]:
display(merged[['review_text', 'cleaned_review_text']].head())

,review_text,cleaned_review_text
0,The overall price was fair for what we paid. M...,the overall price was fair for what we paid mi...
1,The room was nothing out of the ordinary for w...,the room was nothing out of the ordinary for w...
2,"Unfortunately, the pool was outdated — it clea...",unfortunately the pool was outdated it clearly...
3,We were disappointed that the neighborhood was...,we were disappointed that the neighborhood was...
4,The room service was standard hotel fare for w...,the room service was standard hotel fare for w...


In [16]:
merged.to_csv("../data/processed/montra_cleaned_data.csv", index= False)
print("Shape:", merged.shape)

Shape: (35000, 23)
